# Dataclass — Nível Básico

**Objetivo:** entender exatamente o que o `@dataclass` gera, usar defaults com segurança, controlar campos com `field()`,
derivar/validar dados em `__post_init__` e converter/copiar instâncias.

Teoria completa em [`dataclass.md`](dataclass.md). Roteiro:

1. Sem dataclass × com dataclass
2. O que foi gerado (inspecionando a classe)
3. Defaults e a armadilha do default mutável
4. `field()`: `repr`, `compare`, `init`
5. `__post_init__`: campos derivados e validação manual
6. Tipos **não** são validados
7. `asdict`, `astuple`, `replace`
8. Ordenação com `order=True`
9. Resumo

## 1. Sem dataclass × com dataclass

Uma classe de dados escrita à mão precisa de `__init__`, `__repr__` e `__eq__`. Repare no bug clássico:
o `__eq__` abaixo **esqueceu** de comparar `color` — dois carros de cores diferentes são considerados iguais.

In [1]:
class CarManual:
    def __init__(self, name, color):
        self.name = name
        self.color = color

    def __repr__(self):
        return f"{self.__class__.__name__}(name={self.name!r}, color={self.color!r})"

    def __eq__(self, other):
        if not isinstance(other, CarManual):
            return NotImplemented
        return self.name == other.name          # bug: esqueceu color

print(f'CarManual("Saveiro", "amarela") == CarManual("Saveiro", "preta"): {CarManual("Saveiro", "amarela") == CarManual("Saveiro", "preta")}')

CarManual("Saveiro", "amarela") == CarManual("Saveiro", "preta"): True


In [2]:
from dataclasses import dataclass

@dataclass
class Car:
    name: str
    color: str

a = Car("Saveiro", "amarela")
b = Car("Saveiro", "preta")
print(f"a: {a}")
print(f"a == b: {a == b}")
print(f'a == cópia: {a == Car("Saveiro", "amarela")}')

a: Car(name='Saveiro', color='amarela')
a == b: False
a == cópia: True


Mesmo comportamento (corrigido), uma fração do código, e os métodos **nunca dessincronizam** dos campos.

## 2. O que foi gerado
O decorador escreve os métodos como código-fonte e os compila **uma vez**, na definição da classe.
Dá para inspecionar:

In [3]:
import inspect
from dataclasses import fields

print(f'métodos gerados: {[m for m in ("__init__", "__repr__", "__eq__", "__hash__", "__match_args__") if m in Car.__dict__]}')
print(f"assinatura do __init__: {inspect.signature(Car.__init__)}")
print(f"__match_args__: {Car.__match_args__}")
print(f"__hash__: {Car.__hash__}  <- None: eq=True sem frozen torna a classe não-hashable")
print(f"campos: {[(f.name, f.type) for f in fields(Car)]}")

métodos gerados: ['__init__', '__repr__', '__eq__', '__hash__', '__match_args__']
assinatura do __init__: (self, name: str, color: str) -> None
__match_args__: ('name', 'color')
__hash__: None  <- None: eq=True sem frozen torna a classe não-hashable
campos: [('name', <class 'str'>), ('color', <class 'str'>)]


## 3. Defaults e a armadilha do default mutável

Campos com default precisam vir **depois** dos campos sem default (igual a parâmetros de função).

In [4]:
@dataclass
class Car:
    name: str
    color: str
    active: bool = False

print(f'Car("Saveiro", "amarela"): {Car("Saveiro", "amarela")}')
print(f'Car("Saveiro", "amarela", True): {Car("Saveiro", "amarela", True)}')
print(f'Car(name="Gol", color="branco", active=True): {Car(name="Gol", color="branco", active=True)}')   # por nome é mais legível

Car("Saveiro", "amarela"): Car(name='Saveiro', color='amarela', active=False)
Car("Saveiro", "amarela", True): Car(name='Saveiro', color='amarela', active=True)
Car(name="Gol", color="branco", active=True): Car(name='Gol', color='branco', active=True)


Com tipos **mutáveis** (`list`, `dict`, `set`) o Python bloqueia o default direto, porque o mesmo objeto seria
compartilhado por todas as instâncias:

In [5]:
try:
    @dataclass
    class Car:
        name: str
        owners: list = []
except ValueError as e:
    print(f"ValueError: {e}")

ValueError: mutable default <class 'list'> for field owners is not allowed: use default_factory


Para entender o perigo que o Python está evitando, veja o mesmo bug numa classe comum, onde ele **não** é bloqueado:

In [6]:
class Garage:
    def __init__(self, cars=[]):          # default avaliado UMA vez, na definição da função
        self.cars = cars

g1, g2 = Garage(), Garage()
g1.cars.append("Fusca")
print(f"g2 também tem o Fusca: {g2.cars}")
print(f"mesma lista: {g1.cars is g2.cars}")

g2 também tem o Fusca: ['Fusca']
mesma lista: True


A solução é `field(default_factory=...)`: a função é chamada **a cada instância**, gerando um objeto novo.

In [7]:
from dataclasses import field

@dataclass
class Car:
    name: str
    color: str
    active: bool = False
    owners: list[str] = field(default_factory=list)

s2000 = Car("Saveiro", "amarela", True, ["Guilherme da Cruz"])
s2010 = Car("SaveiroCross", "amarela", True)
s2010.owners.append("Guilherme Castro")
print(f"s2000: {s2000}")
print(f"s2010: {s2010}")
print(f"listas independentes: {s2000.owners is not s2010.owners}")

s2000: Car(name='Saveiro', color='amarela', active=True, owners=['Guilherme da Cruz'])
s2010: Car(name='SaveiroCross', color='amarela', active=True, owners=['Guilherme Castro'])
listas independentes: True


`default_factory` também serve para defaults **dinâmicos**:

In [8]:
import uuid
from datetime import datetime, timezone

@dataclass
class Event:
    name: str
    id: str = field(default_factory=lambda: uuid.uuid4().hex[:8])
    created_at: datetime = field(default_factory=lambda: datetime.now(timezone.utc))

print(f'Event("login"): {Event("login")}')
print(f'Event("logout"): {Event("logout")}')

Event("login"): Event(name='login', id='953fb6d7', created_at=datetime.datetime(2026, 11, 11, 17, 11, 12, 303339, tzinfo=datetime.timezone.utc))
Event("logout"): Event(name='logout', id='e254944d', created_at=datetime.datetime(2026, 11, 11, 17, 11, 12, 303942, tzinfo=datetime.timezone.utc))


## 4. `field()`: controlando cada campo

| argumento | efeito |
|---|---|
| `repr=False` | some do `repr` (listas grandes, segredos) |
| `compare=False` | ignorado no `==` (e na ordenação) |
| `init=False` | não é parâmetro do construtor; precisa ser preenchido no `__post_init__` ou ter default |

In [9]:
@dataclass
class Car:
    name: str
    color: str
    active: bool = False
    owners: list[str] = field(default_factory=list, repr=False, compare=False)
    api_token: str = field(default="", repr=False, compare=False)

c1 = Car("Saveiro", "amarela", True, ["Guilherme da Cruz"], api_token="sk-secreto")
c2 = Car("Saveiro", "amarela", True, ["Guilherme Castro"])
print(f"c1: {c1}")                                   # sem owners e sem token
print(f"iguais ignorando donos e token: {c1 == c2}")
print(f'cor diferente ainda conta: {c1 == Car("Saveiro", "preta", True)}')

c1: Car(name='Saveiro', color='amarela', active=True)
iguais ignorando donos e token: True
cor diferente ainda conta: False


## 5. `__post_init__`: campos derivados e validação manual

O `__init__` gerado chama `__post_init__` no final (se existir). É o lugar para:
* **derivar** campos (`init=False`)
* **normalizar** valores
* **validar** regras — lembrando que dataclass não faz isso sozinha

In [10]:
@dataclass
class Car:
    name: str
    color: str
    year: int
    owners: list[str] = field(default_factory=list, repr=False)
    description: str = field(init=False)

    def __post_init__(self):
        self.name = self.name.strip().title()
        if not (1950 <= self.year <= 2027):
            raise ValueError(f"ano inválido: {self.year}")
        self.description = f"{self.name} {self.color} {self.year}"

print(f'Car("  saveiro cross ", "amarela", 2010): {Car("  saveiro cross ", "amarela", 2010)}')
try:
    Car("Fusca", "azul", 1800)
except ValueError as e:
    print(f"ValueError: {e}")
try:
    Car("Fusca", "azul", 1970, description="tentando passar")
except TypeError as e:
    print(f"TypeError: {e}")

Car("  saveiro cross ", "amarela", 2010): Car(name='Saveiro Cross', color='amarela', year=2010, description='Saveiro Cross amarela 2010')
ValueError: ano inválido: 1800
TypeError: Car.__init__() got an unexpected keyword argument 'description'


## 6. Tipos **não** são validados

As anotações só informam ao gerador (e ao editor/mypy) quais são os campos. Em tempo de execução, qualquer valor é aceito:

In [11]:
@dataclass
class Product:
    name: str
    price: float

p = Product(name=123, price="caro")
print(f"p: {p}")
print(f"type(p.name).__name__: {type(p.name).__name__}")
print(f"type(p.price).__name__: {type(p.price).__name__}")

try:
    print(f"p.price * 1.1: {p.price * 1.1}")
except TypeError as e:
    print(f"o erro só aparece longe da origem: {e}")

p: Product(name=123, price='caro')
type(p.name).__name__: int
type(p.price).__name__: str
o erro só aparece longe da origem: can't multiply sequence by non-int of type 'float'


Esse é o principal trade-off: **dataclass é rápida e simples porque confia em quem a cria**.
Para dados externos, valide antes (ou use Pydantic na borda). Um type checker (`mypy`, `pyright`) pega esses erros
**no código**, mas não em dados que chegam em runtime.

**Correção (só stdlib):** validar e converter no `__post_init__`, para o erro aparecer **na criação**, perto da origem:

In [12]:
@dataclass
class Product:
    name: str
    price: float

    def __post_init__(self):
        if not isinstance(self.name, str) or not self.name.strip():
            raise TypeError(f"name deve ser str não vazia, recebido {self.name!r}")
        try:
            self.price = float(self.price)          # aceita "19.9", rejeita "caro"
        except (TypeError, ValueError):
            raise TypeError(f"price deve ser numérico, recebido {self.price!r}") from None
        if self.price < 0:
            raise ValueError(f"price não pode ser negativo: {self.price}")

print(f'Product("Mouse", "19.9"): {Product("Mouse", "19.9")}')
for args in [(123, 10.0), ("Teclado", "caro"), ("Cabo", -5)]:
    try:
        Product(*args)
    except (TypeError, ValueError) as e:
        print(f"Product{args}: {type(e).__name__}: {e}")

Product("Mouse", "19.9"): Product(name='Mouse', price=19.9)
Product(123, 10.0): TypeError: name deve ser str não vazia, recebido 123
Product('Teclado', 'caro'): TypeError: price deve ser numérico, recebido 'caro'
Product('Cabo', -5): ValueError: price não pode ser negativo: -5.0


**Correção (com Pydantic na borda):** a mesma dataclass, validada por um `TypeAdapter` — detalhes em `dataclass/03`, seção 9.

In [13]:
from pydantic import TypeAdapter, ValidationError

@dataclass
class ProductPlain:
    name: str
    price: float

try:
    TypeAdapter(ProductPlain).validate_python({"name": 123, "price": "caro"})
except ValidationError as e:
    print(f"ValidationError: {[(err['loc'], err['type']) for err in e.errors()]}")
print(f'TypeAdapter(ProductPlain).validate_python({{"name": "Mouse", "price": "19.9"}}): {TypeAdapter(ProductPlain).validate_python({"name": "Mouse", "price": "19.9"})}')

ValidationError: [(('name',), 'string_type'), (('price',), 'float_parsing')]
TypeAdapter(ProductPlain).validate_python({"name": "Mouse", "price": "19.9"}): ProductPlain(name='Mouse', price=19.9)


## 7. `asdict`, `astuple`, `replace`

In [14]:
from dataclasses import asdict, astuple, replace

@dataclass
class Address:
    street: str
    city: str

@dataclass
class Customer:
    name: str
    address: Address
    tags: list[str] = field(default_factory=list)

c = Customer("Ana", Address("Rua A, 10", "Recife"), ["vip"])

d = asdict(c)                     # recursivo: Address também vira dict
print(f"d: {d}")
print(f"astuple(c): {astuple(c)}")

moved = replace(c, address=Address("Av. B, 200", "Olinda"))   # cópia com alterações (chama __init__)
print(f"moved: {moved}")
print(f"original intacto: {c.address.city}")

d: {'name': 'Ana', 'address': {'street': 'Rua A, 10', 'city': 'Recife'}, 'tags': ['vip']}
astuple(c): ('Ana', ('Rua A, 10', 'Recife'), ['vip'])
moved: Customer(name='Ana', address=Address(street='Av. B, 200', city='Olinda'), tags=['vip'])
original intacto: Recife


> `asdict` faz **cópia profunda** — alterar o dict não afeta o objeto. `replace` faz cópia **rasa**:
> campos não alterados são compartilhados (`moved.tags is c.tags` → `True`). Cuidado com listas.

In [15]:
print(f"tags compartilhadas após replace: {moved.tags is c.tags}")

# o risco: alterar a lista da cópia altera a do original
moved.tags.append("mudou-se")
print(f"c.tags (original, afetado): {c.tags}")

# correção: passar uma cópia nova dos campos mutáveis no replace (ou copy.deepcopy no objeto todo)
c = Customer("Ana", Address("Rua A, 10", "Recife"), ["vip"])
moved = replace(c, address=Address("Av. B, 200", "Olinda"), tags=list(c.tags))
moved.tags.append("mudou-se")
print(f"c.tags (original, intacto): {c.tags}")
print(f"moved.tags: {moved.tags}")
print(f"moved.tags is c.tags: {moved.tags is c.tags}")

import json
print(f"json.dumps(asdict(c), ensure_ascii=False): {json.dumps(asdict(c), ensure_ascii=False)}")          # caminho simples para JSON

tags compartilhadas após replace: True
c.tags (original, afetado): ['vip', 'mudou-se']
c.tags (original, intacto): ['vip']
moved.tags: ['vip', 'mudou-se']
moved.tags is c.tags: False
json.dumps(asdict(c), ensure_ascii=False): {"name": "Ana", "address": {"street": "Rua A, 10", "city": "Recife"}, "tags": ["vip"]}


## 8. Ordenação com `order=True`

Gera `<`, `<=`, `>`, `>=` comparando os campos **na ordem de declaração** (como tuplas).
Use `compare=False` nos campos que não devem participar.

In [16]:
@dataclass(order=True)
class Version:
    major: int
    minor: int
    patch: int
    label: str = field(default="", compare=False)

versions = [Version(1, 10, 0), Version(1, 2, 3, "lts"), Version(2, 0, 0, "beta"), Version(1, 2, 10)]
print(f"sorted(versions): {sorted(versions)}")
print(f"max: {max(versions)}")
print(f"Version(1, 2, 3) < Version(1, 10, 0): {Version(1, 2, 3) < Version(1, 10, 0)}  <- 2 < 10 numericamente (como string seria o contrário)")

sorted(versions): [Version(major=1, minor=2, patch=3, label='lts'), Version(major=1, minor=2, patch=10, label=''), Version(major=1, minor=10, patch=0, label=''), Version(major=2, minor=0, patch=0, label='beta')]
max: Version(major=2, minor=0, patch=0, label='beta')
Version(1, 2, 3) < Version(1, 10, 0): True  <- 2 < 10 numericamente (como string seria o contrário)


## ⚠️ Pegadinhas, efeitos colaterais e caveats

### Já demonstradas acima

| # | pegadinha | onde | como evitar |
|---|---|---|---|
| 1 | `eq=True` sem `frozen` deixa a classe sem `__hash__` | §2 | `frozen=True` se precisar ir em `set`/`dict` |
| 2 | default mutável (`list`, `dict`, `set`) é recusado | §3 | `field(default_factory=...)` |
| 3 | tipos **não** são validados | §6 | validar na borda (Pydantic / `__post_init__`) |
| 4 | `replace` faz cópia rasa: campos mutáveis ficam compartilhados | §7 | passar cópias novas dos campos mutáveis |

### Novas

**1. A proteção contra default mutável só reconhece tipos *unhashable*.** Uma instância de classe sua passa — e é compartilhada:

In [17]:
class Basket:
    def __init__(self):
        self.items = []


@dataclass
class Checkout:
    basket: Basket = Basket()          # aceito: Basket é hashable (hash por identidade)


first, second = Checkout(), Checkout()
first.basket.items.append("mouse")
print(f"second.basket.items (ninguém mexeu nele): {second.basket.items}")
print(f"mesmo objeto: {first.basket is second.basket}")

second.basket.items (ninguém mexeu nele): ['mouse']
mesmo objeto: True


O `@dataclass` recusa `list`/`dict`/`set` porque eles têm `__hash__ = None`. Qualquer outro objeto mutável passa. Regra simples:
default que **não** for imutável (número, str, tupla, `None`, frozen) vai em `default_factory`.

**2. Campo derivado no `__post_init__` fica velho quando a origem muda.**

In [18]:
@dataclass
class Playlist:
    songs: list[str] = field(default_factory=list)

    def __post_init__(self):
        self.total = len(self.songs)


playlist = Playlist(["a", "b"])
playlist.songs.append("c")
print(f"playlist.total: {playlist.total}")
print(f"len(playlist.songs): {len(playlist.songs)}")

playlist.total: 2
len(playlist.songs): 3


O `__post_init__` roda **uma vez**, na criação. Valor derivado que precisa acompanhar a origem é uma `@property` (calculada a cada
acesso), não um atributo calculado no construtor — ou a classe é `frozen` e a origem é imutável (`tuple`).

**3. `order=True` com `None` num campo quebra a ordenação.**

In [19]:
@dataclass(order=True)
class Job:
    priority: int | None
    name: str


try:
    sorted([Job(1, "backup"), Job(None, "relatório")])
except TypeError as ex:
    print(f"sorted([...]): TypeError: {ex}")

sorted([...]): TypeError: '<' not supported between instances of 'NoneType' and 'int'


A comparação é feita como **tupla de campos**: `(1, ...) < (None, ...)` compara `1 < None`. Funciona nos testes (sem `None`) e
quebra em produção. Use `sorted(..., key=...)` com uma chave explícita (ex.: `lambda j: (j.priority is None, j.priority or 0)`)
em vez de `order=True` quando houver campos opcionais.

**4. `==` exige a mesma classe — subclasse "igual" é diferente.**

In [20]:
@dataclass
class Money:
    cents: int


@dataclass
class Price(Money):
    pass


print(f"Money(100) == Price(100): {Money(100) == Price(100)}")
print(f"Money(100) == Money(100): {Money(100) == Money(100)}")

Money(100) == Price(100): False
Money(100) == Money(100): True


O `__eq__` gerado compara `type(a) is type(b)` antes dos campos. Um teste `assert resultado == Money(100)` falha se a função passar
a devolver uma subclasse. Compare campos (`asdict`) quando a hierarquia não importa.

**5. `asdict` faz cópia profunda — e quebra com o que não é copiável.**

In [21]:
import threading


@dataclass
class Worker:
    name: str
    lock: threading.Lock = field(default_factory=threading.Lock)


try:
    asdict(Worker("w1"))
except TypeError as ex:
    print(f"asdict(Worker(...)): TypeError: {ex}")
print(f"só os campos de dados: {({f.name: getattr(Worker('w1'), f.name) for f in fields(Worker) if f.name != 'lock'})}")

asdict(Worker(...)): TypeError: cannot pickle '_thread.lock' object
só os campos de dados: {'name': 'w1'}


Locks, conexões, arquivos abertos e geradores não são copiáveis. Mantenha recursos fora das dataclasses de **dados**, ou serialize
escolhendo os campos (como acima) em vez de `asdict`.

## 9. Resumo

| conceito | lembrete |
|---|---|
| `@dataclass` | gera `__init__`, `__repr__`, `__eq__` (e mais, sob demanda) uma vez, na definição |
| defaults | sem default antes, com default depois |
| mutáveis | **sempre** `field(default_factory=...)` |
| `field()` | `repr=False`, `compare=False`, `init=False` |
| `__post_init__` | derivar, normalizar e **validar manualmente** |
| tipos | só anotação — nada é validado em runtime |
| utilitários | `asdict` (deep), `astuple`, `replace` (cópia rasa) |
| `order=True` | comparação por tupla de campos |

### Exercícios
1. Crie `Money(amount_cents: int, currency: str)` que valide `currency` em `{"BRL", "USD"}` e rejeite valores negativos.
2. Crie `Task` com `created_at` automático e `compare=False`, e confirme que duas tarefas iguais criadas em momentos diferentes são `==`.
3. Ordene uma lista de `Student(grade: float, name: str)` pela nota e, no empate, pelo nome.

➡️ Próximo: [`02_dataclass_intermediario.ipynb`](02_dataclass_intermediario.ipynb)